# Results overview

A thin view over the files in `results/`. All logic lives in `src/rim`; this notebook only loads summaries and draws figures. It computes no new result.

In [ ]:
import json
from pathlib import Path

import pandas as pd

RESULTS = Path('..') / 'results'
summaries = {p.parent.name: json.loads(p.read_text()) for p in sorted(RESULTS.glob('*/summary.json'))}
list(summaries)

## Retrieval: test metrics per system

In [ ]:
for name, s in summaries.items():
    if 'systems' in s:
        table = pd.DataFrame({k: v['test'] for k, v in s['systems'].items()}).T
        display(table[['ndcg@10', 'recall@20', 'recall@100', 'mrr@10']].round(4).style.set_caption(name))

## Quality against latency

In [ ]:
from rim.viz import plot_quality_latency

for name, s in summaries.items():
    if 'latency_pipelines' not in s:
        continue
    points = []
    for system, lat in s['latency_pipelines'].items():
        if system in s['systems']:
            group = 'reranked' if '+rerank' in system else ('hybrid' if system.startswith('hybrid') else 'single retriever')
            label = system.split('@')[-1] if '+rerank' in system else system
            points.append({'name': label, 'latency_ms': lat['median_ms'], 'quality': s['systems'][system]['test']['ndcg@10'], 'group': group})
    print(plot_quality_latency(points, RESULTS / 'figures' / f'quality_latency_{name}.png'))

## Per-query differences

Paired comparisons are computed from these per-query files.

In [ ]:
per_query = pd.read_csv(RESULTS / 'retrieval_litsearch' / 'per_query_test.csv')
wide = per_query.pivot(index='query_id', columns='system', values='ndcg@10')
(wide['hybrid_convex'] - wide['bm25']).describe()

## Provenance of a run

In [ ]:
run = json.loads((RESULTS / 'retrieval_litsearch' / 'run.json').read_text())
{k: run[k] for k in ('run_id', 'seed', 'git', 'started_at', 'finished_at')}, run['environment']['hardware']